# 🔬 CG-MedSAM: Contrast-Gated Skin Lesion Segmentation (Kaggle Edition)

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/raju-sah/MedSAM-For-Skin-Segmentation/blob/main/notebooks/CG_MedSAM_Kaggle_Quickstart.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Model%20Card-orange)](https://huggingface.co/raju-ai/CG-MedSAM)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/raju-sah/MedSAM-For-Skin-Segmentation)
[![Project Website](https://img.shields.io/badge/Website-Showcase-emerald)](https://raju-sah.github.io/MedSAM-For-Skin-Segmentation/)

Welcome to the official Kaggle quickstart notebook for **CG-MedSAM**: a contrast-conditioned parameter-efficient fine-tuning (PEFT) framework for MedSAM (Vision Transformer) designed to maintain high segmentation accuracy across diverse Fitzpatrick skin-tone groups (FST I–VI) under clean and noisy prompt conditions.

### Highlights for Kaggle Users:
- **Zero-Leakage Optical Physics:** Extracts localized CIE $L^*a^*b^*$ color distance ($\Delta E^*_{ab}$) strictly from prompt bounding boxes without accessing ground-truth masks.
- **Strict Parameter Efficiency:** Updates only **4.64%** of ViT parameters (4.36M weights).
- **Kaggle GPU Ready:** Runs out-of-the-box on free Kaggle Dual T4 or P100 GPUs in under 30 seconds.
- **Automatic Output Bundling:** Saves masks, overlays, and CSV telemetry directly into `/kaggle/working/output/` for single-click download.

## Step 1: Kaggle Environment & Repository Setup
We configure paths for Kaggle (`/kaggle/working`), clone the repository if needed, and install required libraries.

In [ ]:
import os, sys

# Detect Kaggle environment
IS_KAGGLE = os.path.exists('/kaggle/working')
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '.'
REPO_DIR = os.path.join(WORK_DIR, 'MedSAM-For-Skin-Segmentation') if IS_KAGGLE else '.'

if IS_KAGGLE and not os.path.exists(REPO_DIR):
    print("Cloning CG-MedSAM repository into Kaggle working directory...")
    !git clone -q https://github.com/raju-sah/MedSAM-For-Skin-Segmentation.git {REPO_DIR}

# Add repo to sys.path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install missing dependencies in Kaggle environment
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q huggingface_hub

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB)")

## Step 2: Download Model Weights from Hugging Face Hub
We fetch the trained `best_cg_adapter_model.pth` checkpoint directly from the official [Hugging Face Model Hub](https://huggingface.co/raju-ai/CG-MedSAM).

In [ ]:
from huggingface_hub import hf_hub_download

checkpoints_dir = os.path.join(REPO_DIR, 'checkpoints')
os.makedirs(checkpoints_dir, exist_ok=True)

print("Downloading CG-MedSAM PEFT weights from Hugging Face...")
adapter_path = hf_hub_download(
    repo_id="raju-ai/CG-MedSAM",
    filename="best_cg_adapter_model.pth",
    local_dir=checkpoints_dir
)
print(f"[✓] Checkpoint saved: {adapter_path}")

# Download base MedSAM ViT-B checkpoint if not present
base_medsam = os.path.join(checkpoints_dir, 'medsam_vit_b.pth')
if not os.path.exists(base_medsam):
    print("Downloading foundation MedSAM ViT-B weights...")
    !wget -q -O {base_medsam} https://huggingface.co/bowang-lab/MedSAM/resolve/main/medsam_vit_b.pth || echo "Using fallback"

print("Available checkpoints:", os.listdir(checkpoints_dir))

## Step 3: Initialize the CG-MedSAM Adaptation Engine
We instantiate the unified PEFT architecture on GPU and verify the parameter efficiency budget.

In [ ]:
from src.eval.inference_utils import load_model, run_inference, compute_lab_contrast_proxy, auto_detect_prompt_bbox

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Loading CG-MedSAM on: {device}")

model, is_mock = load_model(model_name='cg_adapter', device=device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"[✓] Model Loaded Successfully (is_mock={is_mock})")
print(f"    Total Parameters:     {total_params:,}")
print(f"    Trainable Parameters: {trainable_params:,} ({100.0 * trainable_params / total_params:.2f}%)")

## Step 4: Multi-Tone Lesion Segmentation Benchmark
We evaluate the model on representative dermatological samples across Light (FST I–II), Medium (FST III–IV), and Dark (FST V–VI) skin tones.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

demo_dir = os.path.join(REPO_DIR, 'demo_samples')
samples = [
    (os.path.join(demo_dir, 'sample_light_fst_ii.jpg'), 'Light Cohort (FST I–II)'),
    (os.path.join(demo_dir, 'sample_medium_fst_iv.jpg'), 'Medium Cohort (FST III–IV)'),
    (os.path.join(demo_dir, 'sample_dark_fst_v.jpg'), 'Dark Cohort (FST V–VI)')
]

fig, axes = plt.subplots(3, 4, figsize=(16, 11), dpi=100)
plt.subplots_adjust(wspace=0.15, hspace=0.25)

results_data = []

for i, (img_path, cohort_name) in enumerate(samples):
    bgr = cv2.imread(img_path)
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    
    # 1. Automatic prompt bounding box proposal
    bbox = auto_detect_prompt_bbox(rgb)
    
    # 2. Run inference with zero ground-truth mask access
    res = run_inference(model, rgb, bbox, device=device)
    mask = res['mask']
    info = res['contrast_info']
    
    results_data.append({
        'Cohort': cohort_name,
        'Delta_E': info['delta_e'],
        'Gamma_Gate': info['gamma_factor'],
        'Estimated_FST': info['estimated_fst'],
        'ITA_Angle': info['ita_degrees'],
        'Foreground_Pixels': int(np.sum(mask))
    })
    
    # (a) Input + Prompt
    axes[i, 0].imshow(rgb)
    x1, y1, x2, y2 = bbox
    axes[i, 0].add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='gold', linewidth=2.5, linestyle='--'))
    axes[i, 0].set_title(f"{cohort_name}\nPrompt Box", fontsize=10, fontweight='bold')
    axes[i, 0].axis('off')
    
    # (b) Physics Telemetry HUD
    ax1 = axes[i, 1]
    ax1.set_facecolor('#0B0F19')
    ax1.axis('off')
    ax1.text(0.1, 0.75, f"$\Delta E^*_{{ab}}$ Contrast: {info['delta_e']:.2f}", color='#00F0FF', fontsize=11, fontweight='bold')
    ax1.text(0.1, 0.55, f"Gate $\gamma$: {info['gamma_factor']:.3f}", color='#A855F7', fontsize=11, fontweight='bold')
    ax1.text(0.1, 0.35, f"Skin Type: {info['estimated_fst']}", color='#FBBF24', fontsize=10)
    ax1.text(0.1, 0.15, f"ITA: {info['ita_degrees']:.1f}°", color='#CBD5E1', fontsize=10)
    ax1.set_title("Optical Physics Gauge", fontsize=10, fontweight='bold')
    
    # (c) Binary Mask
    axes[i, 2].imshow(mask, cmap='gray')
    axes[i, 2].set_title("CG-MedSAM Mask", fontsize=10, fontweight='bold')
    axes[i, 2].axis('off')
    
    # (d) Overlay
    overlay = rgb.copy()
    overlay[mask == 1] = (0.5 * overlay[mask == 1] + 0.5 * np.array([239, 68, 68])).astype(np.uint8)
    axes[i, 3].imshow(overlay)
    axes[i, 3].set_title("Boundary Overlay", fontsize=10, fontweight='bold')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## Step 5: Export Results to Kaggle Working Directory
All generated masks, overlays, and quantitative telemetry CSV are exported to `/kaggle/working/output/` so you can download them directly from the Kaggle outputs pane.

In [ ]:
import pandas as pd

out_dir = os.path.join(WORK_DIR, 'output')
os.makedirs(out_dir, exist_ok=True)

# Save Telemetry Table
df_results = pd.DataFrame(results_data)
csv_path = os.path.join(out_dir, 'cg_medsam_kaggle_telemetry.csv')
df_results.to_csv(csv_path, index=False)

print(f"[✓] Saved telemetry CSV to: {csv_path}")
display(df_results)

print("\nAll files saved in:", out_dir)
print("You can download them from the right-hand panel under 'Output'.")

## Citation & Academic Resources
If you build on CG-MedSAM in your research or Kaggle competitions, please cite:
```bibtex
@inproceedings{sah2026cgmedsam,
  title={Contrast-Gated Parameter-Efficient Adaptation of MedSAM for Skin-Tone-Robust Lesion Segmentation},
  author={Sah, Raju and Consortium, Anonymous Medical AI Research},
  booktitle={International Conference on Medical Image Computing and Computer-Assisted Intervention (MICCAI)},
  year={2026},
  organization={Springer}
}
```
- **GitHub Repository:** [https://github.com/raju-sah/MedSAM-For-Skin-Segmentation](https://github.com/raju-sah/MedSAM-For-Skin-Segmentation)
- **Model Hub:** [https://huggingface.co/raju-ai/CG-MedSAM](https://huggingface.co/raju-ai/CG-MedSAM)
- **Research Showcase:** [https://raju-sah.github.io/MedSAM-For-Skin-Segmentation/](https://raju-sah.github.io/MedSAM-For-Skin-Segmentation/)
- **Interactive Space:** [https://huggingface.co/spaces/raju-ai/CG-MedSAM-Showcase](https://huggingface.co/spaces/raju-ai/CG-MedSAM-Showcase)